# Reviderad modellträning för Boprisindikatorn

Filen kan öppnas i VS Code och köras cell för cell som en notebook.
Testdatan används inte innan en slutmodell har valts.



## Importera paket



In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder




## Läs in datasetet

Vi behåller originalet oförändrat och arbetar med en kopia.



In [ ]:
DATA_PATH = Path("../data/SwedenHousingPrices.csv")

df_original = pd.read_csv(Path("../data/SwedenHousingPrices.csv"), encoding="utf-8-sig")
df = df_original.copy()

print(f"Antal rader i rådata: {len(df)}")
df.head()




Antal rader i rådata: 11549


,ad_id,date_published,typology,asking_price_sek,land_area_sqm,living_area_sqm,sqm_price_sek,number_rooms,address,location,coordenates
0,21440984,2024-12-28,APARTMENT,3695000.0,0.0,82.0,45061.0,4.0,Einar Hansens esplanad 14,"Västra Hamnen, Malmö kommun","55.6118431,12.9822884"
1,21462070,2024-12-28,APARTMENT,3095000.0,0.0,109.0,28394.0,4.0,Skarpskyttevägen 30A,"Norra Fäladen, Lunds kommun","55.7230072,13.1973028"
2,21455084,2024-12-28,APARTMENT,2295000.0,0.0,54.0,42500.0,2.0,Boplatsvägen 9,"Brotorp/järvastaden, Sundbybergs kommun","59.381943,17.973993"
3,21411619,2024-12-28,APARTMENT,3675000.0,0.0,119.0,30882.0,4.0,Drottninggatan 34C,"Alingsås, Alingsås kommun","57.93111,12.5369"
4,21461458,2024-12-28,APARTMENT,295000.0,0.0,99.0,2980.0,4.0,Valhallagatan 19A,"Skara, Skara kommun","58.38283,13.4190693"


## Behåll bostadstyperna som appen stödjer



In [ ]:
allowed_typologies = ["APARTMENT", "HOUSE", "ROW_HOUSE"]
df = df[df["typology"].isin(allowed_typologies)].copy()

df["typology"].value_counts()




## Skapa koordinater och kommun

Kommunen hämtas från den sista delen av `location`.



In [ ]:
df[["latitude", "longitude"]] = (
    df["coordenates"]
    .str.split(",", expand=True)
    .astype(float)
)

df["municipality"] = (
    df["location"]
    .str.split(", ")
    .str[-1]
    .str.strip()
)

df[["location", "municipality", "latitude", "longitude"]].head()




## Grundläggande datatvätt

Vi tar bort ogiltiga värden och avgränsar modellen till utgångspriser mellan
100 000 och 10 miljoner kronor.



In [ ]:
rows_before = len(df)

df = df[
    df["asking_price_sek"].between(100_000, 10_000_000)
    & (df["living_area_sqm"] > 0)
    & (df["number_rooms"] > 0)
    & df["latitude"].between(55.0, 69.1)
    & df["longitude"].between(10.5, 24.2)
].copy()

print(f"Borttagna rader: {rows_before - len(df)}")
print(f"Kvarvarande rader: {len(df)}")




## Bostadstypsspecifika gränser

Reglerna avgränsar modellen från extrema specialobjekt.



In [ ]:
apartment_mask = (
    (df["typology"] == "APARTMENT")
    & df["living_area_sqm"].between(10, 300)
    & df["number_rooms"].between(1, 10)
)

house_mask = (
    (df["typology"] == "HOUSE")
    & df["living_area_sqm"].between(25, 400)
    & df["number_rooms"].between(1, 15)
    & (df["land_area_sqm"] <= 4_000)
)

row_house_mask = (
    (df["typology"] == "ROW_HOUSE")
    & df["living_area_sqm"].between(40, 300)
    & df["number_rooms"].between(1, 10)
)

df = df[apartment_mask | house_mask | row_house_mask].copy()

df["typology"].value_counts()




## Hantera tomtarea

Tomtarea används inte för lägenheter. Villavärden under 50 m² är osäkra och
behandlas som saknade i stället för att vi gissar en enhet.



In [ ]:
df["has_land_area"] = (df["land_area_sqm"] > 0).astype(int)

df.loc[
    df["typology"] == "APARTMENT",
    "land_area_sqm"
] = np.nan

df.loc[
    (df["typology"] == "HOUSE")
    & (df["land_area_sqm"] < 50),
    "land_area_sqm"
] = np.nan

print(f"Antal rader efter all tvätt: {len(df)}")




## Dela data i 60 % train, 20 % validation och 20 % test

Först avsätts 20 % till test. Därefter delas återstående data så att 20 % av
hela datasetet blir validation. `stratify` bevarar fördelningen av bostadstyper.



In [ ]:
train_validation_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["typology"]
)

train_df, validation_df = train_test_split(
    train_validation_df,
    test_size=0.25,
    random_state=42,
    stratify=train_validation_df["typology"]
)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(validation_df)}")
print(f"Test: {len(test_df)}")




## Kontrollera fördelningen av bostadstyper

Tabellen visar hur många lägenheter, villor och radhus som finns i varje del.



In [ ]:
split_distribution = pd.DataFrame({
    "Train": train_df["typology"].value_counts(),
    "Validation": validation_df["typology"].value_counts(),
    "Test": test_df["typology"].value_counts()
})

split_distribution




## Funktion för modellernas resultat



In [ ]:
def calculate_metrics(y_true, predictions):
    return {
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "Median error": median_absolute_error(y_true, predictions),
        "R2": r2_score(y_true, predictions)
    }


def prepare_metrics_for_saving(metrics):
    return {
        "mae": float(metrics["MAE"]),
        "rmse": float(metrics["RMSE"]),
        "median_error": float(metrics["Median error"]),
        "r2": float(metrics["R2"])
    }




## Medianbaseline

Baseline gissar alltid medianpriset. ML-modellerna ska slå detta resultat.



In [ ]:
baseline = DummyRegressor(strategy="median")

baseline.fit(
    np.zeros((len(train_df), 1)),
    train_df["asking_price_sek"]
)

baseline_predictions = baseline.predict(
    np.zeros((len(validation_df), 1))
)

baseline_metrics = calculate_metrics(
    validation_df["asking_price_sek"],
    baseline_predictions
)

pd.DataFrame([baseline_metrics], index=["Median baseline"])




## Välj features

Den globala modellen använder bostadstyp. Specialmodellerna behöver inte den
kolumnen eftersom de tränas på en bostadstyp i taget.



In [ ]:
global_numeric_features = [
    "land_area_sqm",
    "living_area_sqm",
    "number_rooms",
    "latitude",
    "longitude",
    "has_land_area"
]

global_categorical_features = ["municipality", "typology"]

apartment_numeric_features = [
    "living_area_sqm",
    "number_rooms",
    "latitude",
    "longitude"
]

property_numeric_features = [
    "land_area_sqm",
    "living_area_sqm",
    "number_rooms",
    "latitude",
    "longitude",
    "has_land_area"
]

specialized_categorical_features = ["municipality"]




## Preprocessing för den globala modellen

Saknade numeriska värden fylls med medianen. Kategorier one-hot-encodas.



In [ ]:
global_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median", add_indicator=True),
        global_numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        global_categorical_features
    )
])




## Preprocessing för lägenheter



In [ ]:
apartment_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        apartment_numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        specialized_categorical_features
    )
])




## Preprocessing för villor och radhus



In [ ]:
property_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median", add_indicator=True),
        property_numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        specialized_categorical_features
    )
])




## Modeller som ska jämföras

Detta är en första screening. Tuning görs efter modelljämförelsen.



In [ ]:
models = {
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=300,
        l2_regularization=1.0,
        random_state=42
    )
}




## Funktion som jämför modellerna

Varje modell tränas på train och utvärderas på validation. Test används inte.



In [ ]:
def compare_models(
    segment_name,
    train_data,
    validation_data,
    feature_columns,
    preprocessor
):
    X_train = train_data[feature_columns]
    y_train = train_data["asking_price_sek"]

    X_validation = validation_data[feature_columns]
    y_validation = validation_data["asking_price_sek"]

    results = []

    for model_name, model in models.items():
        pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", clone(model))
        ])

        pipeline.fit(X_train, y_train)

        train_predictions = pipeline.predict(X_train)
        validation_predictions = pipeline.predict(X_validation)

        train_rmse = np.sqrt(
            mean_squared_error(y_train, train_predictions)
        )

        metrics = calculate_metrics(
            y_validation,
            validation_predictions
        )

        results.append({
            "Segment": segment_name,
            "Model": model_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "Median error": metrics["Median error"],
            "R2": metrics["R2"],
            "Train RMSE": train_rmse,
            "RMSE gap": metrics["RMSE"] - train_rmse
        })

    return pd.DataFrame(results)




## Jämför modeller på hela datasetet



In [ ]:
global_features = global_numeric_features + global_categorical_features

global_results = compare_models(
    "GLOBAL",
    train_df,
    validation_df,
    global_features,
    global_preprocessor
)

global_results.sort_values("RMSE")




## Jämför modeller för lägenheter



In [ ]:
apartment_train = train_df[train_df["typology"] == "APARTMENT"].copy()
apartment_validation = validation_df[
    validation_df["typology"] == "APARTMENT"
].copy()

apartment_features = (
    apartment_numeric_features
    + specialized_categorical_features
)

apartment_results = compare_models(
    "APARTMENT",
    apartment_train,
    apartment_validation,
    apartment_features,
    apartment_preprocessor
)

apartment_results.sort_values("RMSE")




## Jämför modeller för villor



In [ ]:
house_train = train_df[train_df["typology"] == "HOUSE"].copy()
house_validation = validation_df[
    validation_df["typology"] == "HOUSE"
].copy()

property_features = (
    property_numeric_features
    + specialized_categorical_features
)

house_results = compare_models(
    "HOUSE",
    house_train,
    house_validation,
    property_features,
    property_preprocessor
)

house_results.sort_values("RMSE")




## Jämför modeller för radhus



In [ ]:
row_house_train = train_df[
    train_df["typology"] == "ROW_HOUSE"
].copy()
row_house_validation = validation_df[
    validation_df["typology"] == "ROW_HOUSE"
].copy()

row_house_results = compare_models(
    "ROW_HOUSE",
    row_house_train,
    row_house_validation,
    property_features,
    property_preprocessor
)

row_house_results.sort_values("RMSE")




## Samlad modelljämförelse



In [ ]:
comparison_df = pd.concat([
    global_results,
    apartment_results,
    house_results,
    row_house_results
], ignore_index=True)

comparison_df = comparison_df.sort_values(["Segment", "RMSE"])
comparison_df.round(0)




## Bästa modell per segment

Vinnaren är modellen med lägst RMSE på validation-setet.



In [ ]:
best_models = comparison_df.loc[
    comparison_df.groupby("Segment")["RMSE"].idxmin()
]

best_models = best_models[
    ["Segment", "Model", "MAE", "RMSE", "R2", "RMSE gap"]
].sort_values("Segment")

best_models.round(2)

print("\nBästa modell per segment:")
print(best_models.round(2).to_string(index=False))




# Hyperparametertuning

Screeningen visade att HistGradientBoosting var bäst för globalmodellen och
lägenheter. Random Forest var bäst för villor och radhus. Vi tunar därför bara
dessa kombinationer.

`GridSearchCV` testar parametrarna med tre delningar av träningsdatan. Testdata
används fortfarande inte.




## Parametrar för HistGradientBoosting

Vi testar tre learning rates, två trädstorlekar och två minsta lövstorlekar.
Det ger totalt 12 kombinationer.



In [ ]:
hist_parameter_grid = {
    "model__learning_rate": [0.03, 0.05, 0.08],
    "model__max_leaf_nodes": [15, 31],
    "model__min_samples_leaf": [10, 20]
}




## Parametrar för Random Forest

Vi testar två träddjup, tre minsta lövstorlekar och två feature-nivåer.
Antalet träd hålls på 400 för stabila resultat.



In [ ]:
random_forest_parameter_grid = {
    "model__max_depth": [None, 20],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [0.7, 1.0]
}




## Tuna den globala HistGradientBoosting-modellen



In [ ]:
global_hist_pipeline = Pipeline([
    ("preprocessor", clone(global_preprocessor)),
    (
        "model",
        HistGradientBoostingRegressor(
            max_iter=300,
            l2_regularization=1.0,
            random_state=42
        )
    )
])

global_hist_search = GridSearchCV(
    estimator=global_hist_pipeline,
    param_grid=hist_parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

global_hist_search.fit(
    train_df[global_features],
    train_df["asking_price_sek"]
)

global_tuned_predictions = global_hist_search.predict(
    validation_df[global_features]
)

global_tuned_metrics = calculate_metrics(
    validation_df["asking_price_sek"],
    global_tuned_predictions
)

print("Bästa globala parametrar:", global_hist_search.best_params_)
print("Validation RMSE:", round(global_tuned_metrics["RMSE"]))




## Tuna HistGradientBoosting för lägenheter



In [ ]:
apartment_hist_pipeline = Pipeline([
    ("preprocessor", clone(apartment_preprocessor)),
    (
        "model",
        HistGradientBoostingRegressor(
            max_iter=300,
            l2_regularization=1.0,
            random_state=42
        )
    )
])

apartment_hist_search = GridSearchCV(
    estimator=apartment_hist_pipeline,
    param_grid=hist_parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

apartment_hist_search.fit(
    apartment_train[apartment_features],
    apartment_train["asking_price_sek"]
)

apartment_tuned_predictions = apartment_hist_search.predict(
    apartment_validation[apartment_features]
)

apartment_tuned_metrics = calculate_metrics(
    apartment_validation["asking_price_sek"],
    apartment_tuned_predictions
)

print("Bästa lägenhetsparametrar:", apartment_hist_search.best_params_)
print("Validation RMSE:", round(apartment_tuned_metrics["RMSE"]))




## Tuna Random Forest för villor



In [ ]:
house_rf_pipeline = Pipeline([
    ("preprocessor", clone(property_preprocessor)),
    (
        "model",
        RandomForestRegressor(
            n_estimators=400,
            random_state=42,
            n_jobs=-1
        )
    )
])

house_rf_search = GridSearchCV(
    estimator=house_rf_pipeline,
    param_grid=random_forest_parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

house_rf_search.fit(
    house_train[property_features],
    house_train["asking_price_sek"]
)

house_tuned_predictions = house_rf_search.predict(
    house_validation[property_features]
)

house_tuned_metrics = calculate_metrics(
    house_validation["asking_price_sek"],
    house_tuned_predictions
)

print("Bästa villaparametrar:", house_rf_search.best_params_)
print("Validation RMSE:", round(house_tuned_metrics["RMSE"]))




## Tuna Random Forest för radhus



In [ ]:
row_house_rf_pipeline = Pipeline([
    ("preprocessor", clone(property_preprocessor)),
    (
        "model",
        RandomForestRegressor(
            n_estimators=400,
            random_state=42,
            n_jobs=-1
        )
    )
])

row_house_rf_search = GridSearchCV(
    estimator=row_house_rf_pipeline,
    param_grid=random_forest_parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

row_house_rf_search.fit(
    row_house_train[property_features],
    row_house_train["asking_price_sek"]
)

row_house_tuned_predictions = row_house_rf_search.predict(
    row_house_validation[property_features]
)

row_house_tuned_metrics = calculate_metrics(
    row_house_validation["asking_price_sek"],
    row_house_tuned_predictions
)

print("Bästa radhusparametrar:", row_house_rf_search.best_params_)
print("Validation RMSE:", round(row_house_tuned_metrics["RMSE"]))




## Sammanställ resultatet efter tuning



In [ ]:
tuning_results = pd.DataFrame([
    {
        "Segment": "GLOBAL",
        "Model": "HistGradientBoosting",
        **global_tuned_metrics
    },
    {
        "Segment": "APARTMENT",
        "Model": "HistGradientBoosting",
        **apartment_tuned_metrics
    },
    {
        "Segment": "HOUSE",
        "Model": "Random Forest",
        **house_tuned_metrics
    },
    {
        "Segment": "ROW_HOUSE",
        "Model": "Random Forest",
        **row_house_tuned_metrics
    }
])

tuning_results.round(2)

print("\nResultat efter tuning:")
print(tuning_results.round(2).to_string(index=False))




## Jämför före och efter tuning

En tunad modell används bara om den faktiskt förbättrar validation-RMSE.



In [ ]:
before_tuning = best_models[
    ["Segment", "RMSE"]
].rename(columns={"RMSE": "RMSE before tuning"})

after_tuning = tuning_results[
    ["Segment", "RMSE"]
].rename(columns={"RMSE": "RMSE after tuning"})

tuning_comparison = before_tuning.merge(
    after_tuning,
    on="Segment"
)

tuning_comparison["Improvement"] = (
    tuning_comparison["RMSE before tuning"]
    - tuning_comparison["RMSE after tuning"]
)

tuning_comparison.round(0)

print("\nFörändring efter tuning:")
print(tuning_comparison.round(0).to_string(index=False))




# Slutlig träning och test

Modellvalet är nu färdigt. Vi slår ihop train och validation, tränar om de
valda modellerna och använder sedan testdatan en enda gång.



In [ ]:
final_train_df = pd.concat(
    [train_df, validation_df],
    ignore_index=True
)

final_apartment_train = final_train_df[
    final_train_df["typology"] == "APARTMENT"
].copy()

final_house_train = final_train_df[
    final_train_df["typology"] == "HOUSE"
].copy()

final_row_house_train = final_train_df[
    final_train_df["typology"] == "ROW_HOUSE"
].copy()




## Träna den slutliga globalmodellen

Vi använder den bästa tunade HistGradientBoosting-pipelinen.



In [ ]:
final_global_model = clone(global_hist_search.best_estimator_)

final_global_model.fit(
    final_train_df[global_features],
    final_train_df["asking_price_sek"]
)




## Träna den slutliga lägenhetsmodellen



In [ ]:
final_apartment_model = clone(apartment_hist_search.best_estimator_)

final_apartment_model.fit(
    final_apartment_train[apartment_features],
    final_apartment_train["asking_price_sek"]
)




## Träna den slutliga villamodellen

Tuningen förbättrade inte validation-RMSE. Därför behåller vi den enklare
Random Forest-konfigurationen från screeningen.



In [ ]:
final_house_model = Pipeline([
    ("preprocessor", clone(property_preprocessor)),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    )
])

final_house_model.fit(
    final_house_train[property_features],
    final_house_train["asking_price_sek"]
)




## Träna den slutliga radhusmodellen



In [ ]:
final_row_house_model = Pipeline([
    ("preprocessor", clone(property_preprocessor)),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    )
])

final_row_house_model.fit(
    final_row_house_train[property_features],
    final_row_house_train["asking_price_sek"]
)




## Utvärdera slutmodellerna på testdata



In [ ]:
apartment_test = test_df[test_df["typology"] == "APARTMENT"].copy()
house_test = test_df[test_df["typology"] == "HOUSE"].copy()
row_house_test = test_df[test_df["typology"] == "ROW_HOUSE"].copy()

global_test_predictions = final_global_model.predict(
    test_df[global_features]
)

apartment_test_predictions = final_apartment_model.predict(
    apartment_test[apartment_features]
)

house_test_predictions = final_house_model.predict(
    house_test[property_features]
)

row_house_test_predictions = final_row_house_model.predict(
    row_house_test[property_features]
)

global_test_metrics = calculate_metrics(
    test_df["asking_price_sek"],
    global_test_predictions
)

apartment_test_metrics = calculate_metrics(
    apartment_test["asking_price_sek"],
    apartment_test_predictions
)

house_test_metrics = calculate_metrics(
    house_test["asking_price_sek"],
    house_test_predictions
)

row_house_test_metrics = calculate_metrics(
    row_house_test["asking_price_sek"],
    row_house_test_predictions
)

final_test_results = pd.DataFrame([
    {
        "Segment": "GLOBAL",
        "Model": "HistGradientBoosting",
        **global_test_metrics
    },
    {
        "Segment": "APARTMENT",
        "Model": "HistGradientBoosting",
        **apartment_test_metrics
    },
    {
        "Segment": "HOUSE",
        "Model": "Random Forest",
        **house_test_metrics
    },
    {
        "Segment": "ROW_HOUSE",
        "Model": "Random Forest",
        **row_house_test_metrics
    }
])

final_test_results.round(2)

print("\nSlutresultat på testdata:")
print(final_test_results.round(2).to_string(index=False))




## Jämför global och specialiserad modell på samma testbostäder

Den globala modellen utvärderas separat på lägenheter, villor och radhus.
Då jämförs modellerna på exakt samma testobjekt.



In [ ]:
global_apartment_predictions = final_global_model.predict(
    apartment_test[global_features]
)

global_house_predictions = final_global_model.predict(
    house_test[global_features]
)

global_row_house_predictions = final_global_model.predict(
    row_house_test[global_features]
)

global_vs_specialized = pd.DataFrame([
    {
        "Segment": "APARTMENT",
        "Model": "Global",
        **calculate_metrics(
            apartment_test["asking_price_sek"],
            global_apartment_predictions
        )
    },
    {
        "Segment": "APARTMENT",
        "Model": "Specialized",
        **calculate_metrics(
            apartment_test["asking_price_sek"],
            apartment_test_predictions
        )
    },
    {
        "Segment": "HOUSE",
        "Model": "Global",
        **calculate_metrics(
            house_test["asking_price_sek"],
            global_house_predictions
        )
    },
    {
        "Segment": "HOUSE",
        "Model": "Specialized",
        **calculate_metrics(
            house_test["asking_price_sek"],
            house_test_predictions
        )
    },
    {
        "Segment": "ROW_HOUSE",
        "Model": "Global",
        **calculate_metrics(
            row_house_test["asking_price_sek"],
            global_row_house_predictions
        )
    },
    {
        "Segment": "ROW_HOUSE",
        "Model": "Specialized",
        **calculate_metrics(
            row_house_test["asking_price_sek"],
            row_house_test_predictions
        )
    }
])

global_vs_specialized.round(2)

print("\nGlobal mot specialiserad modell på samma testdata:")
print(global_vs_specialized.round(2).to_string(index=False))

# Spara globalmodellens metrics separat för varje bostadstyp.
global_metrics_by_segment = {
    "APARTMENT": prepare_metrics_for_saving(
        calculate_metrics(
            apartment_test["asking_price_sek"],
            global_apartment_predictions
        )
    ),
    "HOUSE": prepare_metrics_for_saving(
        calculate_metrics(
            house_test["asking_price_sek"],
            global_house_predictions
        )
    ),
    "ROW_HOUSE": prepare_metrics_for_saving(
        calculate_metrics(
            row_house_test["asking_price_sek"],
            global_row_house_predictions
        )
    )
}




# Spara modellerna

Varje fil innehåller tre delar:

- `pipeline`: preprocessing och den tränade modellen
- `metrics`: modellens slutresultat på testdatan
- `metadata`: information om hur modellen skapades



In [ ]:
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)




## Skapa bundle för globalmodellen



In [ ]:
global_bundle = {
    "pipeline": final_global_model,
    "metrics": prepare_metrics_for_saving(global_test_metrics),
    "metrics_by_segment": global_metrics_by_segment,
    "metadata": {
        "segment": "GLOBAL",
        "model_name": "HistGradientBoostingRegressor",
        "training_rows": len(final_train_df),
        "test_rows": len(test_df),
        "feature_columns": global_features,
        "target": "asking_price_sek",
        "price_min": 100_000,
        "price_max": 10_000_000,
        "random_state": 42,
        "data_file": DATA_PATH.name,
        "sklearn_version": sklearn.__version__
    }
}




## Skapa bundle för lägenhetsmodellen



In [ ]:
apartment_bundle = {
    "pipeline": final_apartment_model,
    "metrics": prepare_metrics_for_saving(apartment_test_metrics),
    "metadata": {
        "segment": "APARTMENT",
        "model_name": "HistGradientBoostingRegressor",
        "training_rows": len(final_apartment_train),
        "test_rows": len(apartment_test),
        "feature_columns": apartment_features,
        "target": "asking_price_sek",
        "price_min": 100_000,
        "price_max": 10_000_000,
        "random_state": 42,
        "data_file": DATA_PATH.name,
        "sklearn_version": sklearn.__version__
    }
}




## Skapa bundle för villamodellen



In [ ]:
house_bundle = {
    "pipeline": final_house_model,
    "metrics": prepare_metrics_for_saving(house_test_metrics),
    "metadata": {
        "segment": "HOUSE",
        "model_name": "RandomForestRegressor",
        "training_rows": len(final_house_train),
        "test_rows": len(house_test),
        "feature_columns": property_features,
        "target": "asking_price_sek",
        "price_min": 100_000,
        "price_max": 10_000_000,
        "random_state": 42,
        "data_file": DATA_PATH.name,
        "sklearn_version": sklearn.__version__
    }
}




## Skapa bundle för radhusmodellen



In [ ]:
row_house_bundle = {
    "pipeline": final_row_house_model,
    "metrics": prepare_metrics_for_saving(row_house_test_metrics),
    "metadata": {
        "segment": "ROW_HOUSE",
        "model_name": "RandomForestRegressor",
        "training_rows": len(final_row_house_train),
        "test_rows": len(row_house_test),
        "feature_columns": property_features,
        "target": "asking_price_sek",
        "price_min": 100_000,
        "price_max": 10_000_000,
        "random_state": 42,
        "data_file": DATA_PATH.name,
        "sklearn_version": sklearn.__version__
    }
}




## Skriv modellfilerna till models-mappen

`compress=3` minskar filstorleken utan att göra sparandet onödigt långsamt.



In [ ]:
joblib.dump(
    global_bundle,
    MODELS_DIR / "global_model.joblib",
    compress=3
)

joblib.dump(
    apartment_bundle,
    MODELS_DIR / "apartment_model.joblib",
    compress=3
)

joblib.dump(
    house_bundle,
    MODELS_DIR / "house_model.joblib",
    compress=3
)

joblib.dump(
    row_house_bundle,
    MODELS_DIR / "row_house_model.joblib",
    compress=3
)

print("\nSparade modellfiler:")
for model_path in sorted(MODELS_DIR.glob("*_model.joblib")):
    size_mb = model_path.stat().st_size / 1_000_000
    print(f"{model_path.name}: {size_mb:.1f} MB")
